# CURE-Rec — complete notebook execution

This notebook is the **single entry point** for the complete Milestone 1 workflow. Running it in order performs every implemented stage:

1. fetch/load external recommendation data;
2. standardize and audit the evidence level;
3. run all registered CPU recommender models;
4. generate external-data analysis assets;
5. execute all CURE-Sim scenarios and all 64 intervention coalitions;
6. compute exact Shapley values, interaction regions, feasibility sensitivity, and direct robust policy selection;
7. generate every numbered CURE-Sim paper asset, logs, manifests, and decision card.

The external-data stage tests data and recommender-model logic. The CURE-Sim stage is the oracle causal benchmark; the notebook does not misrepresent ordinary ratings data as long-horizon policy-intervention evidence.

## 1. Setup

Install once from `paper-ideas/CURE-Rec/code/`:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[dev]'
jupyter lab notebooks/00_cure_rec_quickstart.ipynb
```

In [13]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from the CURE-Rec code directory or repository root.')

# VS Code kernels are long-lived. Always force this notebook to use the current
# repository source, not a stale cure_rec module imported before a git pull.
sys.path[:] = [str(ROOT), *[entry for entry in sys.path if entry != str(ROOT)]]
for module_name in list(sys.modules):
    if module_name == 'cure_rec' or module_name.startswith('cure_rec.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

from cure_rec.config import load_settings
from cure_rec.data import DatasetLoadResult, load_dataset
from cure_rec.workflow import run_full_workflow

import cure_rec.data as data_layer
assert hasattr(data_layer, 'DatasetLoadResult'), f'Stale data module loaded: {data_layer.__file__}'
print('Project root:', ROOT)
print('Data module:', data_layer.__file__)
print('CURE-Rec data layer: current')


Project root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code
Data module: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/cure_rec/data.py
CURE-Rec data layer: current


## 2. Configure the entire run

`quick` is designed for interactive use. `full` uses the larger CURE-Sim configuration. The external-data fetch is explicit and visible: set `FETCH_IF_MISSING = False` when using already downloaded local data.

In [14]:
# CURE-Sim configuration
CURE_MODE = 'quick'  # quick | full
config_name = 'curesim_quickstart.yaml' if CURE_MODE == 'quick' else 'curesim_full.yaml'
settings = load_settings(ROOT / 'configs' / config_name)

# External-data and registered model configuration
PUBLIC_DATASET = 'movielens_1m'  # movielens_1m | coat | yahoo_r3 | csv
PUBLIC_SOURCE = ROOT / 'data' / 'raw' / PUBLIC_DATASET
FETCH_IF_MISSING = True  # explicit opt-in network retrieval for MovieLens-1M / Coat
RUN_BPR_MF = True
BPR_UPDATES = 50_000 if CURE_MODE == 'quick' else 200_000
MAX_EVAL_USERS = 1_000

print('CURE config:', config_name, '| hash:', settings.config_hash())
print('CURE users/items/horizon:', settings.simulator.n_users, settings.simulator.n_items, settings.simulator.horizon)
print('CURE interventions:', list(settings.interventions.costs))
print('CURE scenarios:', [scenario.name for scenario in settings.scenarios])
print('External dataset:', PUBLIC_DATASET, '| source:', PUBLIC_SOURCE)


CURE config: curesim_quickstart.yaml | hash: a15ab769c9aab911
CURE users/items/horizon: 24 72 4
CURE interventions: ['repeat_cap', 'explore_slot', 'tail_slot', 'diversify', 'novel_slot', 'provider_balance']
CURE scenarios: ['nominal', 'fatigue_stress', 'popularity_stress']
External dataset: movielens_1m | source: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/data/raw/movielens_1m


## 3. Run every implemented stage

This one cell calls the full orchestrator. It **fetches/loads data first**, audits the evidence, runs popularity and BPR-MF baselines when chronology is available, and then runs the full CURE-Sim causal workflow and asset generator.

In [15]:
workflow = run_full_workflow(
    settings,
    dataset=PUBLIC_DATASET,
    source=PUBLIC_SOURCE,
    download=FETCH_IF_MISSING,
    run_bpr=RUN_BPR_MF,
    bpr_updates=BPR_UPDATES,
    max_eval_users=MAX_EVAL_USERS,
)

public_result = workflow.dataset
data_analysis = workflow.analysis
logger = workflow.logger
game = workflow.game
RUN_DIR = workflow.cure_run_dir
decision = workflow.decision

print('External-data analysis run:', data_analysis.run_dir)
print('External-data evidence level:', data_analysis.audit.permitted_claim)
print('CURE-Rec run:', RUN_DIR)
print('Decision:', decision.action)
print('Selected portfolio:', decision.selected_interventions)
print('Worst-case improvement:', round(decision.lower_improvement, 5))


2026-08-04 15:44:48,969 | INFO | run_started | {"config_hash": "a15ab769c9aab911", "run_id": "curesim-quickstart-20260804T144448Z-827cbde1"}
2026-08-04 15:44:48,969 | INFO | exact_game_started | {}
2026-08-04 15:44:48,970 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "nominal"}
2026-08-04 15:44:54,369 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.4210904357912536, "scenario": "nominal", "shapley_efficiency_gap": 5.551115123125783e-17}
2026-08-04 15:44:54,369 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "fatigue_stress"}
2026-08-04 15:44:59,720 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.4187732306444988, "scenario": "fatigue_stress", "shapley_efficiency_gap": 5.551115123125783e-17}
2026-08-04 15:44:59,721 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "popularity_stress"}
2026-08-04 15:45:05,173 | INFO | scenario_game_comple

## 4. Inspect loaded data, audit logic, and every registered baseline model

These results describe the external interaction data. They are intentionally separate from the CURE-Sim causal results below.

In [16]:
print('Loader metadata:')
print(public_result.metadata)
print('Audit notes:', *data_analysis.audit.notes, sep='\n- ')
display(data_analysis.summary)
display(data_analysis.model_metrics)

print('External-data assets:')
for path in sorted((data_analysis.run_dir / 'tables').glob('*.csv')):
    print('-', path.name)
for path in sorted((data_analysis.run_dir / 'figures').glob('*.png')):
    print('-', path.name)


Loader metadata:
{'ratings_path': '/Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/data/raw/movielens_1m/ml-1m/ratings.dat', 'rows': 1000209, 'has_exposure_log': False}
Audit notes:
- Missing logged slate/propensity fields; do not make offline causal-policy claims.


,dataset,users,items,interactions,positive_interactions,positive_rate,density,timestamps_available
0,movielens_1m,6040,3706,1000209,575281,0.575161,0.044684,True


,model,evaluated_users,recall_at_k,ndcg_at_k,hit_rate_at_k
0,popularity,1000,0.049,0.025520,0.049
1,bpr_mf,1000,0.007,0.004932,0.007


External-data assets:
- data_table_item_activity.csv
- data_table_model_metrics.csv
- data_table_summary.csv
- data_table_user_activity.csv
- data_figure_activity_distributions.png
- data_figure_model_metrics.png


## 5. Inspect the complete CURE-Rec causal game

The game uses all six interventions, exact coalition values, exact Shapley contributions, feasibility-aware semivalue sensitivity, and Grabisch–Roubens pairwise interactions. Direct robust improvement—not a sum of Shapley lower endpoints—selects the portfolio.

In [17]:
display(game.regions.sort_values('phi_mean', ascending=False))
display(game.interaction_table.sort_values('interaction_mean', ascending=False))

coalitions = game.coalition_table.groupby('mask', as_index=False).agg(
    lower_improvement=('improvement', 'min'),
    upper_improvement=('improvement', 'max'),
    cost=('cost', 'first'),
    interventions=('active_interventions', 'first'),
).sort_values('lower_improvement', ascending=False)
display(coalitions.head(12))


,intervention,phi_lower,phi_upper,phi_mean,psi_feasible_lower,psi_feasible_upper,phi_psi_sign_agree
0,repeat_cap,0.019328,0.021651,0.020355,0.019107,0.021458,True
3,diversify,-0.058360,-0.057684,-0.057936,-0.057792,-0.056812,True
2,tail_slot,-0.079257,-0.078018,-0.078605,-0.080382,-0.078750,True
4,novel_slot,-0.092236,-0.087646,-0.089787,-0.093722,-0.088783,True
1,explore_slot,-0.099465,-0.098286,-0.098737,-0.100854,-0.099472,True
5,provider_balance,-0.116113,-0.114098,-0.115354,-0.116218,-0.114792,True


,intervention_i,intervention_j,interaction_lower,interaction_upper,interaction_mean
14,novel_slot,provider_balance,0.002719,0.004190,0.003328
12,diversify,novel_slot,0.002373,0.003395,0.002879
5,explore_slot,tail_slot,0.001386,0.003722,0.002804
7,explore_slot,novel_slot,0.002158,0.003382,0.002772
3,repeat_cap,novel_slot,0.000554,0.001605,0.001177
6,explore_slot,diversify,0.000240,0.001268,0.000872
10,tail_slot,novel_slot,-0.000909,0.001235,0.000504
9,tail_slot,diversify,-0.000561,0.000884,0.000109
2,repeat_cap,diversify,-0.000787,-0.000456,-0.000576
8,explore_slot,provider_balance,-0.001096,-0.000162,-0.000596


,mask,lower_improvement,upper_improvement,cost,interventions
1,1,0.027033,0.032587,0.05,repeat_cap
0,0,0.000000,0.000000,0.00,
9,9,-0.032752,-0.027082,0.11,repeat_cap;diversify
5,5,-0.055457,-0.052778,0.13,repeat_cap;tail_slot
8,8,-0.060242,-0.058317,0.06,diversify
17,17,-0.067962,-0.064039,0.13,repeat_cap;novel_slot
4,4,-0.077633,-0.073657,0.08,tail_slot
3,3,-0.079463,-0.073913,0.15,repeat_cap;explore_slot
33,33,-0.092927,-0.089431,0.17,repeat_cap;provider_balance
16,16,-0.096463,-0.092377,0.08,novel_slot


## 6. Inspect all numbered paper assets, logs, and manifests

Every CURE-Sim run generates Tables 1–8, Figures 1–8, an asset registry, per-coalition manifests, JSONL events, raw coalition values, and a deployment/explanation decision card.

In [18]:
asset_manifest = json.loads((RUN_DIR / 'artifacts' / 'asset_manifest.json').read_text())
asset_registry = pd.DataFrame(asset_manifest)
display(asset_registry)

print('Generated CURE tables:')
for path in sorted((RUN_DIR / 'tables').glob('*.csv')):
    print('-', path.name)
print('\nGenerated CURE figures:')
for path in sorted((RUN_DIR / 'figures').glob('*.png')):
    print('-', path.name)

events = pd.DataFrame([json.loads(line) for line in (RUN_DIR / 'logs' / 'events.jsonl').read_text().splitlines()])
display(events[['timestamp_utc', 'event']].tail(20))

decision_card = json.loads((RUN_DIR / 'artifacts' / 'explanation_card.json').read_text())
decision_card


,exists,id,path,purpose,scope
0,False,Table 1,tables/table_01_asset_registry.csv,"Asset provenance, scope, and readiness",generated
1,True,Table 2,tables/table_02_benchmark_configuration.csv,CURE-Sim and policy configuration,generated
2,True,Table 3,tables/table_03_attribution_regions.csv,Full-game Shapley and feasibility-aware semiva...,generated
3,True,Table 4,tables/table_04_uncertainty_summary.csv,Scenario uncertainty widths and attribution signs,generated
4,True,Table 5,tables/table_05_portfolio_decision.csv,Robust selected portfolio and constraint diagn...,generated
5,True,Table 6,tables/table_06_long_term_tradeoffs.csv,Base versus selected policy outcomes by scenario,generated
6,True,Table 7,tables/table_07_selection_comparison.csv,"Base, best-single, full, and robust portfolio ...",generated
7,True,Table 8,tables/table_08_runtime_summary.csv,Coalition evaluation runtime by scenario and c...,generated
8,True,Figure 1,figures/figure_01_framework.png,CURE-Rec execution flow,generated
9,True,Figure 2,figures/figure_02_shapley_regions.png,Shapley regions and selected interventions,generated


Generated CURE tables:
- coalition_values.csv
- interaction_regions.csv
- shapley_regions.csv
- table_01_asset_registry.csv
- table_02_benchmark_configuration.csv
- table_03_attribution_regions.csv
- table_04_uncertainty_summary.csv
- table_05_portfolio_decision.csv
- table_06_long_term_tradeoffs.csv
- table_07_selection_comparison.csv
- table_08_runtime_summary.csv

Generated CURE figures:
- figure_01_framework.png
- figure_02_shapley_regions.png
- figure_03_uncertainty_widths.png
- figure_04_interaction_heatmap.png
- figure_05_trajectory_comparison.png
- figure_06_decision_card.png
- figure_07_runtime_by_cardinality.png
- figure_08_scenario_sensitivity.png


,timestamp_utc,event
391,2026-08-04T14:45:05.173164+00:00,scenario_game_completed
392,2026-08-04T14:45:05.177958+00:00,exact_game_completed
393,2026-08-04T14:45:05.178322+00:00,robust_planning_started
394,2026-08-04T14:45:05.179088+00:00,portfolio_rejected
395,2026-08-04T14:45:05.179578+00:00,portfolio_rejected
396,2026-08-04T14:45:05.179933+00:00,portfolio_rejected
397,2026-08-04T14:45:05.180279+00:00,portfolio_rejected
398,2026-08-04T14:45:05.180597+00:00,portfolio_rejected
399,2026-08-04T14:45:05.180917+00:00,portfolio_rejected
400,2026-08-04T14:45:05.181255+00:00,portfolio_rejected


{'decision': {'action': 'deploy',
  'cost': 0.05,
  'fatigue_upper': 0.0,
  'feasible': True,
  'lower_improvement': 0.02703258221131133,
  'provider_disparity_upper': 0.22428385416666674,
  'reason': 'Selected by exact direct maximin improvement across configured scenarios.',
  'relevance_delta_lower': -0.03932083801222208,
  'selected_interventions': ['repeat_cap'],
  'selected_mask': 1,
  'upper_improvement': 0.03258652756926306},
 'interpretation': {'abstention': 'A non-positive robust improvement returns the base policy unchanged.',
  'positive_certificate': 'phi_lower > 0 means positive order-averaged marginal contribution across configured scenarios.',
  'selection_rule': 'Portfolio selection used direct robust improvement, not summed Shapley lower bounds.'},
 'rejected_or_deferred': [{'intervention': 'explore_slot',
   'phi_lower': -0.09946490999477074,
   'phi_mean': -0.09873716288602752,
   'phi_psi_sign_agree': True,
   'phi_upper': -0.09828633454740567,
   'psi_feasible_low

## 7. Evidence and runtime notes

- MovieLens, Coat, Yahoo! R3, and generic ratings CSVs are loaded and audited before baseline-model analysis. Their audit result controls the permitted scientific claim.
- CURE-Sim is the full causal/oracle environment in this milestone.
- Real long-horizon policy/OPE assets remain gated until an audited slate-policy log and sequential estimator are implemented.
- For a larger CURE-Sim run, change `CURE_MODE = 'full'` and rerun this notebook from the top.
- The same workflow is available from the terminal via `cure-rec full-run`.